<a href="https://www.kaggle.com/code/asivakumarnair/diabetic-retinopathy-imagenet?scriptVersionId=343314517" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ===== FULL REBUILD, NEW SESSION: environment through Stage 11, APTOS, CUSTOM CNN ONLY =====

!pip install -q tensorflow==2.19.0

import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import random
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, MaxPooling2D,
                                      Dropout, GlobalAveragePooling2D, Dense)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split

# ---------- CONFIG ----------
APTOS_CSV     = '/kaggle/input/competitions/aptos2019-blindness-detection/train.csv'
APTOS_IMG     = '/kaggle/input/competitions/aptos2019-blindness-detection/train_images'
EYEPACS_CSV   = '/kaggle/input/datasets/benjaminwarner/resized-2015-2019-blindness-detection-images/labels/trainLabels15.csv'
EYEPACS_IMG   = '/kaggle/input/datasets/benjaminwarner/resized-2015-2019-blindness-detection-images/resized train 15'
MESSIDOR_CSV  = '/kaggle/input/datasets/mariaherrerot/messidor2preprocess/messidor_data.csv'
MESSIDOR_IMG  = '/kaggle/input/datasets/mariaherrerot/messidor2preprocess/messidor-2/messidor-2/preprocess'

GRADES      = ['0','1','2','3','4']
IMG_SIZE, BATCH_SIZE = 224, 32
SUBSAMPLE_SEED = 42
EYEPACS_TARGET = 3662
CUSTOM_LR, EARLYSTOP_PAT, MONITOR = 1e-3, 7, 'val_accuracy'
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)

# ---------- DATA REBUILD (all three sources loaded, since disjointness/pairing gates need all three) ----------
aptos = pd.read_csv(APTOS_CSV)
aptos['grade']      = aptos['diagnosis'].astype(int).astype(str)
aptos['image_path'] = APTOS_IMG + '/' + aptos['id_code'].astype(str) + '.png'
aptos['source']     = 'aptos'
aptos['patient_id'] = None

eyepacs = pd.read_csv(EYEPACS_CSV)
eyepacs['grade']      = eyepacs['level'].astype(int).astype(str)
eyepacs['image_path'] = EYEPACS_IMG + '/' + eyepacs['image'].astype(str) + '.jpg'
eyepacs['source']     = 'eyepacs'
eyepacs['patient_id'] = eyepacs['image'].str.extract(r'^(\d+)_')

messidor = pd.read_csv(MESSIDOR_CSV)
messidor['grade']      = messidor['diagnosis'].astype(int).astype(str)
messidor['image_path'] = MESSIDOR_IMG + '/' + messidor['id_code'].astype(str)
messidor['source']     = 'messidor'
messidor['patient_id'] = None

def subsample_eyepacs(df, target_n=EYEPACS_TARGET, seed=SUBSAMPLE_SEED):
    pg = df.groupby('patient_id')['grade'].max().reset_index()
    frac = target_n / len(df)
    keep, _ = train_test_split(pg, train_size=frac, stratify=pg['grade'], random_state=seed)
    return df[df['patient_id'].isin(keep['patient_id'])].reset_index(drop=True)

eyepacs_s = subsample_eyepacs(eyepacs)

def safe_split(df, label_col, test_size, rs, tag=""):
    try:
        return train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=rs)
    except ValueError as e:
        print(f"WARNING [{tag}]: stratified split failed, falling back to unstratified.")
        return train_test_split(df, test_size=test_size, random_state=rs)

def split_image_level(df, rs=SEED, tag=""):
    tr, tmp = safe_split(df, 'grade', 0.30, rs, tag=f"{tag} first")
    va, te  = safe_split(tmp, 'grade', 0.50, rs, tag=f"{tag} second")
    return tr, va, te

# ---------- ONLY APTOS split is actually needed for this cell ----------
a_tr, a_va, a_te = split_image_level(aptos, tag="APTOS")

def get_source_class_weight(source_train_df):
    cls = np.array(GRADES)
    cw = compute_class_weight('balanced', classes=cls, y=source_train_df['grade'])
    return {i: w for i, w in enumerate(cw)}

aptos_class_weight = get_source_class_weight(a_tr)
span = max(aptos_class_weight.values())/min(aptos_class_weight.values())
print(f"\nAPTOS class weight span: {span:.1f}x (expect ~9.4x)")

# ================================================================
# STAGE 11: APTOS, CUSTOM CNN ONLY
# ================================================================

def make_source_gens(tr_df, va_df, te_df):
    train_idg = ImageDataGenerator(rescale=1./255, **AUG)
    eval_idg  = ImageDataGenerator(rescale=1./255)
    common = dict(x_col='image_path', y_col='grade', target_size=(IMG_SIZE,IMG_SIZE),
                  batch_size=BATCH_SIZE, class_mode='categorical', classes=GRADES, color_mode='rgb')
    tr = train_idg.flow_from_dataframe(tr_df, shuffle=True,  seed=SEED, **common)
    va = eval_idg.flow_from_dataframe(va_df,  shuffle=False, **common)
    te = eval_idg.flow_from_dataframe(te_df,  shuffle=False, **common)
    return tr, va, te

def build_custom_cnn(num_classes=5, shape=(224,224,3)):
    return Sequential([
        Input(shape=shape),
        Conv2D(32,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(32,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        Conv2D(64,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(64,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        Conv2D(128,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(128,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        GlobalAveragePooling2D(),
        Dense(256,activation='relu'), Dropout(0.5),
        Dense(num_classes,activation='softmax')
    ])

tag = "ss_custom_aptos_dr"
tr, va, te = make_source_gens(a_tr, a_va, a_te)
model = build_custom_cnn()
model.compile(Adam(CUSTOM_LR), 'categorical_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=False)])
cbs = [EarlyStopping(monitor=MONITOR, patience=EARLYSTOP_PAT, restore_best_weights=True),
       ModelCheckpoint(f'/kaggle/working/{tag}.keras', monitor=MONITOR, save_best_only=True),
       CSVLogger(f'/kaggle/working/{tag}_log.csv', append=False)]

print(f"\n===== Custom CNN, APTOS (single source, isolation control, weight span {span:.1f}x) =====")
model.fit(tr, validation_data=va, epochs=60, class_weight=aptos_class_weight, callbacks=cbs, verbose=1)
result = model.evaluate(te, verbose=0)
print(f"\n{tag} TEST: loss={result[0]:.4f} accuracy={result[1]:.4f} auc={result[2]:.4f}")

pd.DataFrame([{'arch':'custom','source':'aptos','loss':result[0],'accuracy':result[1],'auc':result[2]}]) \
  .to_csv('/kaggle/working/dr_stage11_custom_aptos.csv', index=False)
print(f"Saved: {tag}.keras, {tag}_log.csv, dr_stage11_custom_aptos.csv")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 59.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
ydf-tf 2.20.0 requires tensorflow==2.20.0, but you have tensorflow 2.19.0 which is incompatible.
tf-keras 2.20.0 requires tensorflow<2.21,>=2.20, but you have tensorflow 2.19.0 which is incompatible.
tensorflow-text 2.20.1 requires tensorflow<2.21,>=2.20.0, but you have tensorflow 2.19.0 which is incompatible.


2026-08-18 20:27:44.653293: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787084864.680346      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787084864.688949      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1787084864.710424      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787084864.710462      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787084864.710466      23 computation_placer.cc:177] computation placer alr

Seed 42 set, TF 2.19.0, tf.keras module: tf_keras.api._v2.keras

APTOS class weight span: 9.4x (expect ~9.4x)
Found 2563 validated image filenames belonging to 5 classes.
Found 549 validated image filenames belonging to 5 classes.
Found 550 validated image filenames belonging to 5 classes.


I0000 00:00:1787084883.454130      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1787084883.461118      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5



===== Custom CNN, APTOS (single source, isolation control, weight span 9.4x) =====
Epoch 1/60


E0000 00:00:1787084890.936242      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential/dropout/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1787084896.306499      75 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1787084899.661798      74 service.cc:152] XLA service 0x7ca9ad224260 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1787084899.661854      74 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1787084899.661861      74 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1787084899.856495      74 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


81/81 [==============================] - 567s 7s/step - loss: 1.5535 - accuracy: 0.4198 - auc: 0.7166 - val_loss: 2.9598 - val_accuracy: 0.1002 - val_auc: 0.2613
Epoch 2/60
81/81 [==============================] - 398s 5s/step - loss: 1.3968 - accuracy: 0.5139 - auc: 0.8120 - val_loss: 2.8614 - val_accuracy: 0.0546 - val_auc: 0.2419
Epoch 3/60
81/81 [==============================] - 399s 5s/step - loss: 1.3182 - accuracy: 0.5447 - auc: 0.8411 - val_loss: 2.8650 - val_accuracy: 0.1002 - val_auc: 0.3058
Epoch 4/60
81/81 [==============================] - 399s 5s/step - loss: 1.2890 - accuracy: 0.5533 - auc: 0.8583 - val_loss: 2.4532 - val_accuracy: 0.0838 - val_auc: 0.2619
Epoch 5/60
81/81 [==============================] - 393s 5s/step - loss: 1.2631 - accuracy: 0.5673 - auc: 0.8616 - val_loss: 2.4539 - val_accuracy: 0.0984 - val_auc: 0.2946
Epoch 6/60
81/81 [==============================] - 390s 5s/step - loss: 1.2794 - accuracy: 0.5899 - auc: 0.8643 - val_loss: 1.5509 - val_accuracy